In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [ ]:
#load the dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()


In [ ]:
#preprocess the data
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [ ]:
##ENCODE CATEGORICAL VARIABLE
label_encode_gender=LabelEncoder()
data['Gender']=label_encode_gender.fit_transform(data['Gender'])
data

In [ ]:
#OHE Georgraphy
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder() 
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder=geo_encoder.toarray()
geo_encoder

In [ ]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

In [ ]:
geo_encoded_df=pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

In [ ]:
## COMBINE ALL THE COLUMNS OHE WITH THE OG DATA
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)


In [ ]:
data.head()


In [ ]:
## SAVE THE ENCODERS AND SCALER
with open('label_encode_gender.pkl','wb') as file:
    pickle.dump(label_encode_gender,file)

In [ ]:
with open('onehot_encoder.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [ ]:
## DIVIDE THE DATA SET INTO INDEPENDENT AND DEPENDENT FEATURES

X=data.drop('Exited',axis=1)
y=data['Exited']

##SPLIT THE DATA IN TRAINING AND TESTING SETS
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#SCALING THESE FEATURES
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
with open ('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
(X_train.shape[1],)

In [ ]:
##Build our ANN MODEL
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected with input layer
    Dense(32,activation='relu'), ##HL2
    Dense(1,activation='sigmoid') ##OUTPUT LAYER
]

)


In [ ]:
model.summary()

In [ ]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()

In [ ]:
## COMPILE THE MODEL TO FORWARD AND BACKWARD PROPAGATION
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [ ]:
##SETUP FOR TENSORBOARD
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="logs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
##SETUP EARLY STOPPING
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
### TRAINING THE MODEL
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
##LOAD TENSORBOARD EXTENSION
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit